# InfoFrota AI: Text-to-SQL em português com RAG de esquema e guarda somente leitura

Notebook de portfólio, **sem GPU e sem chave de API**. Mostra as partes do projeto que rodam de forma determinística:
o banco sintético, a recuperação de contexto (RAG de esquema com BM25), a guarda de SQL e os resultados do benchmark.

**O que este notebook não faz:** não chama nenhum modelo de linguagem e não treina nada. O SQL das demonstrações foi escrito à mão.
Os dados são **sintéticos** (só os rótulos das linhas vêm do CSV aberto da CETURB-ES) e os números do benchmark são de um experimento
com 56 perguntas, não de operação real.

Repositório: https://github.com/David-Evora/InfoFrota (licença MIT, projeto independente, sem endosso institucional).

In [ ]:
# Requer Internet ligada nas configurações do notebook
!git clone -q https://github.com/David-Evora/InfoFrota.git
%cd InfoFrota
!python scripts/generate_data.py

## 1. O banco sintético

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("file:data/infofrota.db?mode=ro", uri=True)
pd.read_sql("SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY 1", con)

In [ ]:
pd.read_sql("""SELECT l.numero, l.nome, COUNT(*) AS alertas
FROM alertas a JOIN linhas l ON l.id = a.linha_id
WHERE a.tipo = 'CRITICO' GROUP BY l.numero ORDER BY alertas DESC LIMIT 5""", con)

## 2. RAG de esquema (BM25)
Para cada pergunta, o índice recupera tabelas, valores válidos e regras do domínio e os coloca no prompt.
O índice usa só o banco e as regras; o benchmark nunca entra nele (há teste automatizado para isso).

In [ ]:
import sys; sys.path.insert(0, ".")
from infofrota_ai.rag import Indice
idx = Indice("data/infofrota.db")
for pergunta in ["Quais linhas têm mais alertas críticos?", "Qual a ocupação média quando chove?"]:
    ctx = idx.recuperar(pergunta)
    print(pergunta)
    print("  tabelas:", [d.id for d in ctx["tabelas"]])
    print("  regras :", [d.texto[:90] for d in ctx["regras"][:2]], "\n")

## 3. Guarda de SQL somente leitura
Toda consulta gerada passa por três camadas: instrução única `SELECT`/`WITH`, *authorizer* do SQLite e conexão `mode=ro`, com limite de 200 linhas e 5 s.

In [ ]:
from infofrota_ai.db import executar, SQLBloqueado
ok = executar("SELECT COUNT(*) AS linhas FROM linhas", "data/infofrota.db")
print("permitido:", ok.colunas, ok.linhas)
for sql in ["DELETE FROM alertas", "SELECT 1; DROP TABLE linhas", "PRAGMA writable_schema = 1"]:
    try:
        executar(sql, "data/infofrota.db"); print("PASSOU (inesperado):", sql)
    except SQLBloqueado as e:
        print("bloqueado:", sql, "->", e)

## 4. Resultados do benchmark congelado (56 perguntas)
Valores copiados de `docs/experiments.md` e de `evaluation/runs/`. Modelo: Llama-3.1-8B-Instruct, 0 exemplos no prompt, critério "o resultado contém o esperado".
Com 56 perguntas, diferenças de um ou dois acertos são ruído. A segunda rodada foi planejada depois de analisar os erros da primeira.

In [ ]:
import matplotlib.pyplot as plt
rodadas = {
    "Prompt compacto": [("Base", 19), ("Base + RAG", 25), ("LoRA", 36), ("LoRA + RAG", 36)],
    "Prompt ultra":    [("Base", 19), ("Base + RAG", 28), ("LoRA (c/ contexto)", 33), ("LoRA (c/ contexto) + RAG", 39)],
}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for ax, (nome, itens) in zip(axs, rodadas.items()):
    ax.barh([n for n, _ in itens][::-1], [v / 56 * 100 for _, v in itens][::-1], color="#2563EB")
    for i, (n, v) in enumerate(itens[::-1]): ax.text(v / 56 * 100 + 1, i, f"{v}/56", va="center")
    ax.set_title(nome); ax.set_xlim(0, 100); ax.set_xlabel("% de acertos")
plt.tight_layout(); plt.show()

## 5. Limitações
- Base sintética e 56 perguntas: os números não medem desempenho em operação real.
- O nível difícil segue baixo (3 de 15 no melhor caso).
- O modelo usado na demonstração da API (Groq) não é o modelo medido no benchmark.
- Os adaptadores LoRA não estão no repositório.